In [ ]:
repository_filter: list[str] = []

In [ ]:
import pandas as pd
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import warnings

warnings.simplefilter("ignore")

df = dt.read_csv("../samples/test_quality_issues.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    language_colors = {
        "java": "#2196F3",
        "javascript": "#FDD835",
        "python": "#4CAF50",
    }

    issue_order = df.groupby("issueType").size().sort_values(ascending=False).index.tolist()

    grouped = df.groupby(["issueType", "language"]).size().reset_index(name="count")

    fig = go.Figure()

    for lang in sorted(grouped["language"].unique()):
        subset = grouped[grouped["language"] == lang]
        lang_counts = dict(zip(subset["issueType"], subset["count"]))
        counts = [lang_counts.get(it, 0) for it in issue_order]
        fig.add_trace(
            go.Bar(
                x=issue_order,
                y=counts,
                name=lang,
                marker_color=language_colors.get(lang, "#999"),
                hovertemplate=(
                    "<b>%{x}</b><br>"
                    f"Language: {lang}<br>"
                    "Count: %{y}"
                    "<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title="Test Quality Issues by Language",
        xaxis_title="Issue Type",
        yaxis_title="Count",
        barmode="group",
        height=500,
        margin=dict(l=60, r=50, t=60, b=120),
        xaxis=dict(tickangle=-45, categoryorder="array", categoryarray=issue_order),
    )

fig.show()